# Batch: save tagged-unit figures (multi-session)

Batch driver around the single-session `save_tagged_unit_figures` notebook. It runs the
full pipeline — **compute metrics -> select tagged units -> render per-unit figures** — for a
**list of sessions**, and lets you choose where the parallelism goes:

- **`PARALLEL_OVER = "sessions"`** — process whole sessions concurrently (one process per
  session, rendering sequentially inside). Best when you have **many sessions**. Uses the
  top-level `opto_tagging_session_worker.run_session` so each session is isolated and a
  failure in one never kills the others.
- **`PARALLEL_OVER = "units"`** — process sessions **one at a time**, fanning the per-unit
  figure rendering across worker processes (the original single-session behaviour). Best for
  **a single large session**.

All compute / selection / plotting knobs are shared across sessions (set once in step 2).
The pulse-aligned panels are annotated with each session's **own** metrics (via a
`"__SELF_METRICS__"` sentinel resolved inside the worker).

In [12]:
# =============================================================================
# 1. ENVIRONMENT SETUP
# =============================================================================
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

print(f"✅ Analysis modules loaded from: {MODULE_PATH}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✅ Analysis modules loaded from: /root/capsule/src/aind_dft_ephys_analysis


In [13]:
# =============================================================================
# 2. BATCH CONFIG (shared across all sessions)
# =============================================================================
# ---- Sessions to process (session_date strings) -----------------------------

SESSIONS = [
"858802_2026-08-13_14-12-20",
"858802_2026-08-12_14-51-04",
"858398_2026-08-05_13-25-22",
"858398_2026-08-04_13-48-57",
"856493_2026-07-23_15-12-36",
"856493_2026-07-22_15-40-13"
]
SESSIONS = [
"858802_2026-08-13_14-12-20",
"858802_2026-08-12_14-51-04",
"872548_2026-09-30_14-08-00",
"872548_2026-10-01_14-13-15",
"868666_2026-09-23_13-47-45",
"860083_2026-09-17_14-46-45",
"863015_2026-09-11_14-36-55",
"863015_2026-09-10_14-36-05",
"863015_2026-09-09_14-22-17",
"863973_2026-08-26_14-55-23"
]



SESSIONS = [
"858802_2026-08-13_14-12-20",
"858802_2026-08-12_14-51-04",
]

# ---- Where the parallelism goes ---------------------------------------------
#   "sessions" -> run whole sessions in parallel (recommended for many sessions)
#   "units"    -> one session at a time, parallel per-unit rendering
PARALLEL_OVER = "sessions"
SESSION_WORKERS = 3    # processes when PARALLEL_OVER == "sessions"
UNIT_WORKERS = 4       # processes when PARALLEL_OVER == "units"

# ---- Output roots -----------------------------------------------------------
DATA_ROOT = "/root/capsule/data"
OUT_ROOT = "/root/capsule/scratch/tagged_units"
METRICS_ROOT = "/root/capsule/scratch/opto_tagging"
SAVE_FORMATS = ["png"]

# If a session's metrics CSV already exists, reuse it instead of recomputing
# (loads the CSV and skips the expensive compute_tagging_metrics step; figures
# are still rendered from the reused metrics). Set False to always recompute.
SKIP_EXISTING_METRICS = True

# ---- Metric CALCULATION params (compute_tagging_metrics) --------------------
COMPUTE_KWARGS = dict(
    baseline_window=(-0.05, 0.0),
    latency_window=(0.0, 0.006),
    salt_dt=0.001,
    per_pulse=True,
    align_to_event="pulse",
    remove_artefacts=True,
    removal_window=0.001,
)

# ---- SELECTION thresholds (select_tagged_units) -----------------------------
SELECT_KWARGS = dict(
    alpha=0.05,
    effect_ratio=2.0,
    min_abs_increase=0.0,
    min_reliability=0.2,
    max_latency=0.006,
    max_jitter=0.003,
    correction="fdr_bh",
)

# ---- Condition(s) to select on (laser_name at index 2 is IGNORED) -----------
#   (target_power, location_tag, laser_name, cycle_duration, frequency, pulse_duration)
# TARGET_COND may be a single 6-tuple OR a list of candidate 6-tuples. Each
# session matches whichever candidate it actually has, so the same config
# covers sessions that use different powers (e.g. 5.0 mW vs 10.0 mW).
TARGET_COND = [
    (5.0,  1.0, "Laser_1", 1.0, 10.0, 0.05),
    (10.0, 1.0, "Laser_1", 1.0, 10.0, 0.05),
    (5.0,  0.0, "Laser_1", 1.0, 10.0, 0.05),
    (10.0, 0.0, "Laser_1", 1.0, 10.0, 0.05),
]
SELECT_PULSE_INDEX = -1
IGNORE_LASER_IDX = 2

# ---- Per-alignment plot kwargs (forwarded to plot_raster_graph) -------------
# "__SELF_METRICS__" is replaced inside the worker by that session's metrics_df.
PULSE_PLOT_KWARGS = dict(
    bin_size=0.002,
    time_window=[-0.02, 0.09],
    removal_window=0.002,
    align_to_event="pulse",
    remove_artefacts=True,
    show_waveform=True,
    metrics_df="__SELF_METRICS__",
)
TRAIN_PLOT_KWARGS = dict(
    bin_size=0.005,
    time_window=[-0.1, 1.2],
    removal_window=0.002,
    align_to_event="laser",
    remove_artefacts=True,
    show_waveform=True,
)

print(f"target_cond: {TARGET_COND} | pulse_index={SELECT_PULSE_INDEX}")
print(f"{len(SESSIONS)} session(s) | parallel over: {PARALLEL_OVER}")


target_cond: [(5.0, 1.0, 'Laser_1', 1.0, 10.0, 0.05), (10.0, 1.0, 'Laser_1', 1.0, 10.0, 0.05), (5.0, 0.0, 'Laser_1', 1.0, 10.0, 0.05), (10.0, 0.0, 'Laser_1', 1.0, 10.0, 0.05)] | pulse_index=-1
2 session(s) | parallel over: sessions


In [14]:
# =============================================================================
# 3. BUILD PER-SESSION CONFIGS (resolve file paths + output dirs)
# =============================================================================
from opto_tagging_session_worker import resolve_session_paths


def build_cfg(session_date):
    beh, nwb = resolve_session_paths(session_date, data_root=DATA_ROOT)
    return dict(
        session_date=session_date,
        data_root=DATA_ROOT,
        behavior_json_file=beh,
        ephys_nwb_file=nwb,
        metrics_csv_path=f"{METRICS_ROOT}/metrics_{session_date}.csv",
        save_metrics=True,
        skip_existing_metrics=SKIP_EXISTING_METRICS,
        compute_kwargs=COMPUTE_KWARGS,
        select_kwargs=SELECT_KWARGS,
        target_cond=TARGET_COND,
        select_pulse_index=SELECT_PULSE_INDEX,
        ignore_laser_idx=IGNORE_LASER_IDX,
        save_formats=SAVE_FORMATS,
        alignments={
            "pulse_aligned": dict(
                out_dir=f"{OUT_ROOT}/{session_date}/pulse_aligned/",
                plot_kwargs=PULSE_PLOT_KWARGS,
            ),
            "train_aligned": dict(
                out_dir=f"{OUT_ROOT}/{session_date}/train_aligned/",
                plot_kwargs=TRAIN_PLOT_KWARGS,
            ),
        },
    )


cfgs = []
for s in SESSIONS:
    try:
        cfgs.append(build_cfg(s))
        print(f"  ok   {s}")
    except FileNotFoundError as e:
        print(f"  SKIP {s}: {e}")
print(f"\n{len(cfgs)} session config(s) ready.")

  ok   858802_2026-08-13_14-12-20
  ok   858802_2026-08-12_14-51-04

2 session config(s) ready.


In [15]:
# =============================================================================
# 4. RUN THE BATCH
# =============================================================================
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor

summaries = []

if PARALLEL_OVER == "sessions":
    # ---- whole sessions in parallel; each renders sequentially inside -------
    from opto_tagging_session_worker import run_session

    ctx = mp.get_context("spawn")  # spawn avoids NWB/HDF5 fork-safety issues
    print(f"Running {len(cfgs)} session(s) across {SESSION_WORKERS} process(es)...")
    with ProcessPoolExecutor(max_workers=SESSION_WORKERS, mp_context=ctx) as ex:
        for res in ex.map(run_session, cfgs):
            summaries.append(res)
            n = res["n_units"]
            al = {k: v["saved"] for k, v in res.get("alignments", {}).items()}
            print(f"  [{res['status']:>12}] {res['session_date']}: "
                  f"{n} unit(s) | saved={al} | err={res['error']}")

elif PARALLEL_OVER == "units":
    # ---- one session at a time; parallel per-unit rendering ----------------
    from opto_tagging_session_worker import run_session  # reuse compute+select
    from opto_tagging_figure_worker import init_worker, render_unit

    ctx = mp.get_context("spawn")
    for cfg in cfgs:
        sd = cfg["session_date"]
        print(f"\n=== {sd} ===")
        # Compute + select in-process (no rendering) by passing empty alignments.
        sel_cfg = dict(cfg, alignments={})
        res = run_session(sel_cfg)
        common_units = res["common_units"]
        print(f"  status={res['status']} | tagged units: {len(common_units)}")
        if not common_units:
            summaries.append(res)
            continue
        # Render each alignment with a per-unit pool.
        import pandas as pd
        metrics_df = pd.read_csv(cfg["metrics_csv_path"])
        res["alignments"] = {}
        for name, spec in cfg["alignments"].items():
            out_dir = spec["out_dir"]
            plot_kwargs = dict(spec["plot_kwargs"])
            for k, v in list(plot_kwargs.items()):
                if isinstance(v, str) and v == "__SELF_METRICS__":
                    plot_kwargs[k] = metrics_df
            init_args = (cfg["behavior_json_file"], cfg["ephys_nwb_file"],
                         out_dir, plot_kwargs, SAVE_FORMATS)
            print(f"  rendering {len(common_units)} '{name}' figures -> {out_dir}")
            saved, failed = 0, 0
            with ProcessPoolExecutor(max_workers=UNIT_WORKERS, mp_context=ctx,
                                     initializer=init_worker, initargs=init_args) as ex:
                for unit_id, ok, msg in ex.map(render_unit, common_units):
                    saved += int(ok); failed += int(not ok)
                    if not ok:
                        print(f"    unit {unit_id} FAILED: {msg}")
            res["alignments"][name] = dict(out_dir=out_dir, saved=saved, failed=failed)
        summaries.append(res)
else:
    raise ValueError(f"PARALLEL_OVER must be 'sessions' or 'units', got {PARALLEL_OVER!r}")

print(f"\nDone. Processed {len(summaries)} session(s).")

Running 2 session(s) across 3 process(es)...


[858802_2026-08-12_14-51-04] loading session (OpticalTagging)...
[858802_2026-08-13_14-12-20] loading session (OpticalTagging)...
Behavior data successfully loaded from: /root/capsule/data/ecephys_858802_2026-08-13_14-12-20/behavior/858802_2026-08-13_14-12-20.json
Behavior data successfully loaded from: /root/capsule/data/ecephys_858802_2026-08-12_14-51-04/behavior/858802_2026-08-12_14-51-04.json


In [ ]:
# =============================================================================
# 5. BATCH SUMMARY TABLE
# =============================================================================
import pandas as pd

rows = []
for r in summaries:
    al = r.get("alignments", {})
    rows.append(dict(
        session_date=r["session_date"],
        status=r["status"],
        n_units=r["n_units"],
        target_exists=r.get("target_exists"),
        matched_targets=r.get("matched_targets"),
        available_powers=r.get("available_powers"),
        pulse_saved=al.get("pulse_aligned", {}).get("saved"),
        train_saved=al.get("train_aligned", {}).get("saved"),
        error=r.get("error"),
    ))
summary_df = pd.DataFrame(rows)
display(summary_df)

,session_date,status,n_units,target_exists,matched_targets,available_powers,pulse_saved,train_saved,error
0,858802_2026-08-13_14-12-20,no_condition,0,False,[],"[0.5, 2.0, 5.0]",NaN,NaN,None
1,858802_2026-08-12_14-51-04,no_condition,0,False,[],"[0.5, 2.0, 5.0]",NaN,NaN,None
2,872548_2026-09-30_14-08-00,ok,88,True,"[(10.0, 1.0, Laser_1, 1.0, 10.0, 0.05)]","[0.5, 2.0, 10.0]",88.0,88.0,None
3,872548_2026-10-01_14-13-15,ok,96,True,"[(10.0, 1.0, Laser_1, 1.0, 10.0, 0.05)]","[0.5, 2.0, 10.0]",96.0,96.0,None
4,868666_2026-09-23_13-47-45,ok,35,True,"[(5.0, 1.0, Laser_1, 1.0, 10.0, 0.05)]","[0.5, 2.0, 5.0]",35.0,35.0,None
5,860083_2026-09-17_14-46-45,ok,61,True,"[(5.0, 1.0, Laser_1, 1.0, 10.0, 0.05)]","[0.5, 2.0, 5.0]",61.0,61.0,None
6,863015_2026-09-11_14-36-55,ok,61,True,"[(5.0, 1.0, Laser_1, 1.0, 10.0, 0.05)]","[0.5, 2.0, 5.0]",61.0,61.0,None
7,863015_2026-09-10_14-36-05,ok,42,True,"[(5.0, 1.0, Laser_1, 1.0, 10.0, 0.05)]","[0.5, 2.0, 5.0]",42.0,42.0,None
8,863015_2026-09-09_14-22-17,ok,34,True,"[(5.0, 1.0, Laser_1, 1.0, 10.0, 0.05)]","[0.5, 2.0, 5.0]",34.0,34.0,None
9,863973_2026-08-26_14-55-23,ok,44,True,"[(5.0, 1.0, Laser_1, 1.0, 10.0, 0.05)]","[0.5, 2.0, 5.0]",44.0,44.0,None
